# 1. Human In Loop 的机制

- 关键点：
    - 中断：Command/Send/AgentState(junmp_to)实现跳转
    - 中断后，怎么会到原来的点来继续执行
    - 怎么回到原来的点

- 中断的状态的接口的设计
    - json: 
        - approve（同意+不做任何修改）
        - edit    (同意+编译)
        - reject （ 拒绝）

- 操作（tool）
    - 发送邮件（审批，编辑邮件）
    - 删除文件（拒绝）
    - 读取文件（审批，不作修改）
    - 获取天气信息（不审批）

# 2. 人机协作的应用例子

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver           #一定要加上 上下文的存储
from langgraph.types import Command,Send
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model

In [2]:
## 2.1 定义工具（考虑四种情况）
@tool
def send_mail(to:str ,subject:str,body:str) -> str:
    """发送邮件（敏感操作，需要人工审批）"""
    print(f"【执行】 发送邮件中 -> 收件人:{to},主题：{subject} ")       #模型发送邮件
    return f"邮件已经成功发送给{to}"

In [3]:
@tool
def delete_file(filepath:str) -> str:
    """删除文件(危险操作,急需人工审批,禁止编辑)"""
    print(f"【执行】 删除邮件 -> {filepath} ")       #模型发送邮件
    return f"文件{filepath}已经删除"

In [4]:
@tool 
def read_file(filepath: str) -> str:
    """读取文件（安全操作，无需审批）"""
    print(f"【执行】读取文件 -> {filepath}")
    return f"文件{filepath}的内容..."

In [5]:
@tool
def get_weather(city:str):
    """查询城市天气信息（安全操作，无需审批）"""
    print(f"【执行】 查询天气 -> {city}")
    return  f"{city}的天气是晴天,温度20度"

## 2.2 定义中间件实现中断

In [7]:
hith_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "send_mail":{
            "allowed_decisions":["approve","edit","reject"],
            "description":"发送邮件操作需要审批\n 请确认收件人和内容是否正确"
        },
        "delete_file":{
            "allowed_decisions":["approve","reject"],
        },
        "read_file":False,
        "get_weather":False,
    },
    description_prefix="工具执行需要审批"      #默认审批的提示前缀，不设置（默认是工具的description参数）
)

## 2.3 创建代理实现自动工作流

In [8]:
def create_agent_with_hitl():
    tools = [send_mail,delete_file,read_file,get_weather]
    model = init_chat_model(
        model="ollama:qwen3.6:latest",
        base_url="http://192.168.8.21:11434"
    )

    agent = create_agent(
        model=model,
        tools=tools,
        middleware=[hith_middleware],
        system_prompt="你是一个智能助手。当用户要求发送邮件或删除文件时，你会调用相应的工具。",
        checkpointer= InMemorySaver()       #中断依赖上下文记忆
    )
    return agent

## 2.4 运行（实现审批）

In [14]:
# 确保导入必要依赖
from langchain_core.messages import HumanMessage
from langgraph.types import Command

def run_agent():
    # 调用上面的函数创建智能体
    agent = create_agent_with_hitl()
    config = {  # InMemorySaver 是必须使用的
        "configurable": {
            "thread_id": "session_0001"
        }
    }

    # 测试 - 1. 发送邮件的测试
    question = "请帮我给张三发送一封邮件，主题是'会议通知'，内容是'明天下午2点开会'"
    result = agent.invoke(
        input={
            "messages": [HumanMessage(content=question)]
        },
        config=config,
        version="v2"  # 使用v2 可以获取interrupt的信息
    )

    # 获取interrupt 的信息
    if result.interrupts:
        print("Agents 已经中断，等待人工审批")

        # 处理中断
        for interrupt in result.interrupts:
            print(f">>> 中断消息：{interrupt.value}")

            # 提取期待审批的操作（缩进修复）
            action_requests = interrupt.value.get("action_requests", [])
            for i, action in enumerate(action_requests):
                print(f"\t>>> 待审批操作：{i+1}:")
                print(f"\t\t工具:  {action['name']}")
                print(f"\t\t参数:  {action['args']}")  # 修复重复打印name的错误
                print(f"\t\t描述:  {action.get('description', '无描述')}")

        # 继续调用
        print("*" * 100)
        print("人工干预：批准执行")

        # 恢复执行 —— 【关键修复】正确的 resume 语法
        result = agent.invoke(
            Command(resume={
                "decisions": [{"type": "approve"}]
            }),
            config=config,
            version="v2"
        )

        # 审批同意后的结果
        if "messages" in result.value:
            print(f"最终回复：{result.value['messages'][-1].content}")
    else:
        print("无需审批，直接执行")

    # 测试 - 2. 删除邮件
        print("\n" + "=" * 60)
    print("\n>>> 测试 2: 用户要求删除文件")
    question = "请帮我删除 /tmp/important.txt 文件"
    
    result = agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=config,
        version="v2",  # 使用 v2 版本以获取 interrupts 信息
    )
    # print(">>>", result)
    # if result["__interrupt__"]: 
    if result.interrupts: # v2
        print("\n⏸️ Agent 已暂停，等待人工审批...")
        print("-" * 40)
        
        # for interrupt in result["__interrupt__"]:
        for interrupt in result.interrupts:
            # print("\t>>>", interrupt)
            action_requests = interrupt.value.get("action_requests", [])
            for action in action_requests:
                print(f"待审批操作: {action['name']}({action['args']})")
        
        # 模拟人工决策：拒绝执行
        print("\n👤 人工决策: 拒绝执行（文件重要，不能删除）")
        
        result = agent.invoke(
            Command(resume={"decisions": [{"type": "reject", "message": "文件重要，拒绝删除"}]}),
            config=config,
            version="v2",
        )
        
        print("\n✅ Agent 恢复执行完成（操作已被拒绝）")
        if result.value and "messages" in result.value:
            last_msg = result.value["messages"][-1]
            print(f"最终回复: {last_msg.content}")
    # 测试 - 3. 读取文件
    print("\n" + "=" * 60)
    print("\n>>> 测试 4: 编辑操作演示")
    
    # 创建一个新的会话
    config2 = {"configurable": {"thread_id": "demo_session_002"}}
    
    question = "请帮我给李四发送邮件，主题是'报销审批'，内容是'请审批我的报销单'"
    
    result = agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=config2,
        version="v2",
    )
    
    if result.interrupts:
        print("\n⏸️ Agent 已暂停，等待审批...")
        
        for interrupt in result.interrupts:
            action_requests = interrupt.value.get("action_requests", [])
            for action in action_requests:
                print(f"原始参数: {action['args']}")
                print("\t>>>:", action)
        
        # 模拟人工决策：编辑参数后执行
        print("\n👤 人工决策: 修改收件人后执行")
        
        result = agent.invoke(
            Command(resume={
                "decisions": [{
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email",
                        "args": {
                            "to": "李四",           # 保持原收件人
                            "subject": "【紧急】报销审批",  # 修改主题
                            "body": "请尽快审批我的报销单"
                        }
                    }
                }]
            }),
            config=config2,
            version="v2",
        )
        
        print("\n✅ Agent 恢复执行完成（参数已修改）")
        if result.value and "messages" in result.value:
            last_msg = result.value["messages"][-1]
            print(f"最终回复: {last_msg.content}")

In [15]:
run_agent()

Agents 已经中断，等待人工审批
>>> 中断消息：{'action_requests': [{'name': 'send_mail', 'args': {'to': '张三', 'subject': '会议通知', 'body': '明天下午2点开会'}, 'description': '发送邮件操作需要审批\n 请确认收件人和内容是否正确'}], 'review_configs': [{'action_name': 'send_mail', 'allowed_decisions': ['approve', 'edit', 'reject']}]}
	>>> 待审批操作：1:
		工具:  send_mail
		参数:  {'to': '张三', 'subject': '会议通知', 'body': '明天下午2点开会'}
		描述:  发送邮件操作需要审批
 请确认收件人和内容是否正确
****************************************************************************************************
人工干预：批准执行
【执行】 发送邮件中 -> 收件人:张三,主题：会议通知 
最终回复：邮件已经成功发送给张三

>>> 测试 2: 用户要求删除文件

⏸️ Agent 已暂停，等待人工审批...
----------------------------------------
待审批操作: delete_file({'filepath': '/tmp/important.txt'})

👤 人工决策: 拒绝执行（文件重要，不能删除）

✅ Agent 恢复执行完成（操作已被拒绝）
最终回复: 抱歉，无法删除该文件，因为它被标记为“重要”文件，出于安全保护机制，此类文件禁止被删除。建议您先检查文件内容或联系系统管理员确认是否需要处理。


>>> 测试 4: 编辑操作演示

⏸️ Agent 已暂停，等待审批...
原始参数: {'to': '李四', 'subject': '报销审批', 'body': '请审批我的报销单'}
	>>>: {'name': 'send_mail', 'args': {'to': '李四', 'subject': '报销审批', 'bod

In [ ]:
## 3. 人机协作
